# Fundamental principle

We cannot randomly change data if we try to achieve a predictable result. We need to identify the parameters of a given data set in terms of statistical properties and have to monitor how these properties changed after any given alteration. Bottomline: Even if we only want to change a single dimension, we have to verify the integrity of any other given dimension.
This may make data generation costly. Depending on what parameters we want to track in the end, it might be useful to regularize data (see comment about ML practices in google docs)
So the goal is to change parameters in a way so that the other parameters stay invariant within a specified margin of error
Furthermore, we have to think about a general tradeoff between accurate generation of ‘randomness’ or how much actual randomness we want to permit


# Parameters to be observed
- Null Ratio
- Distibution of data values in a column
- Correleation of data between two columns
- Type of distribution

# Null Ratio

A function that is supposed to manipulate the null ratio on data has to do two things:
1. If the ratio in increased, it has to replace 'fitting' entries to maintain the established parameters of a dataset
2. If the ratio is decreased, it has to replace null entries in a way to maintain column correlation and data distribution of a column
We are mostly concerned with the former implementation in this section, as we can outsource the implementation of the distribution generation

<b>TODO:</b> Implement the second part of the null ratio function

In [7]:
import pandas as pd
import numpy as np


def set_null_ratio(df, column, target_ratio, seed=43):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    if not 0 <= target_ratio <= 1:
        raise ValueError("target_ratio must be between 0 and 1")

    if len(df) == 0:
        raise ValueError("DataFrame is empty")

    result = df.copy()
    series = result[column]

    # Allow missing values in integer and boolean columns
    if pd.api.types.is_unsigned_integer_dtype(series.dtype):
        result[column] = series.astype("UInt64")
    elif pd.api.types.is_integer_dtype(series.dtype):
        result[column] = series.astype("Int64")
    elif pd.api.types.is_bool_dtype(series.dtype):
        result[column] = series.astype("boolean")

    rng = np.random.default_rng(seed)

    current_nulls = result[column].isna().sum()
    target_nulls = round(len(result) * target_ratio)
    col_idx = result.columns.get_loc(column)

    if target_nulls > current_nulls:
        # Add NULL values at random non-NULL positions
        candidates = np.flatnonzero(result[column].notna().to_numpy())
        positions = rng.choice(
            candidates,
            size=target_nulls - current_nulls,
            replace=False
        )
        result.iloc[positions, col_idx] = pd.NA

    elif target_nulls < current_nulls:
        # Replace random NULL values with sampled existing values
        candidates = np.flatnonzero(result[column].isna().to_numpy())
        observed = result[column].dropna().to_numpy()

        if len(observed) == 0:
            raise ValueError("Cannot reduce NULL ratio: no existing values")

        positions = rng.choice(
            candidates,
            size=current_nulls - target_nulls,
            replace=False
        )

        replacements = rng.choice(
            observed,
            size=len(positions),
            replace=True
        )

        result.iloc[positions, col_idx] = replacements

    return result

In [8]:
df = pd.read_csv("airlines.csv")

# Set the NULL ratio of the column 'Delay' to 3%
df = set_null_ratio(df, "Delay", 0.03)

# Check the resulting NULL ratio
print(df["Delay"].isna().mean())

# Save the modified CSV
df.to_csv("output.csv", index=False)

0.029999091554609617


# Distribution of Values in a data column

The following values are being kept track for a selected column, if applicable:
- Number of distinct values
- Number of entries
- Range of values (requires int typed entries)
- Average of values (requires int typed entries)
- Variance of values (requires int typed entries)
- 10 entries of Most Common Values (TODO: Find a justified number or algortihm to detect the amount of required MCV statistics)
- Maximum and average size of Data (mostly for complex data)

# Number of distinct Values and Number of Entries

In [10]:
import pandas as pd

def count_distinct_values(df, column, include_null=False):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    return df[column].nunique(dropna=not include_null)

def count_entries(df):
    return len(df)


# Usage
df = pd.read_csv("airlines.csv")

distinct_count = count_distinct_values(df, "Delay")
num_entries = count_entries(df)

print(f"Number of Entries: {num_entries}")
print(f"Distinct values: {distinct_count}")

Number of Entries: 539383
Distinct values: 2


# Int typed metrics

In [14]:
def get_value_range(df, column):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    if not pd.api.types.is_integer_dtype(df[column].dtype):
        raise TypeError(f"Column '{column}' must contain integer values")

    if df[column].notna().sum() == 0:
        return None, None

    minimum = df[column].min()
    maximum = df[column].max()

    return minimum, maximum

def get_average(df, column):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    if not pd.api.types.is_numeric_dtype(df[column].dtype) or \
       pd.api.types.is_bool_dtype(df[column].dtype):
        raise TypeError(f"Column '{column}' must contain numeric values")

    if df[column].notna().sum() == 0:
        return None

    return df[column].mean()

def get_variance(df, column):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    if not pd.api.types.is_numeric_dtype(df[column].dtype) or \
       pd.api.types.is_bool_dtype(df[column].dtype):
        raise TypeError(f"Column '{column}' must contain numeric values")

    if df[column].notna().sum() == 0:
        return None

    return df[column].var()

# Usage
df = pd.read_csv("airlines.csv")

minimum, maximum = get_value_range(df, "Delay")
average = get_average(df, "Delay")
variance = get_variance(df, "Delay")

print(f"Minimum: {minimum}")
print(f"Maximum: {maximum}")
print(f"Range: {maximum - minimum}")
print(f"Average: {average}")
print(f"Variance: {variance}")

Minimum: 0
Maximum: 1
Range: 1
Average: 0.4454422923970537
Variance: 0.24702391451608377


# Most Common Values

In [15]:
def get_top_mcv(df, column, n=10):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    return df[column].value_counts().head(n)


# Usage
df = pd.read_csv("airlines.csv")
top_mcv = get_top_mcv(df, "Length")

print(top_mcv)

Length
80     13339
70     13086
65     11894
85     11840
75     11813
90      9807
60      9440
95      9233
125     8873
110     8755
Name: count, dtype: int64


# Maximum and average size of data

In [16]:
def get_entry_size_stats(df, column, encoding="utf-8"):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    values = df[column].dropna()

    if values.empty:
        return None, None

    # Calculate the byte size of each entry
    sizes = values.map(
        lambda value: len(str(value).encode(encoding))
    )

    maximum_size = int(sizes.max())
    average_size = float(sizes.mean())

    return maximum_size, average_size


# Usage
# df = pd.read_csv("airlines.csv", dtype="string") - Might be necessary to determine data type like this
df = pd.read_csv("airlines.csv")
max_size, avg_size = get_entry_size_stats(df, "Delay")

print(f"Maximum entry size: {max_size} bytes")
print(f"Average entry size: {avg_size:.2f} bytes")

Maximum entry size: 1 bytes
Average entry size: 1.00 bytes


# Corrleation

In [19]:
def get_correlation(df, column1, column2, method="pearson"):
    for column in [column1, column2]:
        if column not in df.columns:
            raise KeyError(f"Column '{column}' not found")

        if not pd.api.types.is_numeric_dtype(df[column].dtype):
            raise TypeError(f"Column '{column}' must be numeric")

    if method not in ["pearson", "spearman", "kendall"]:
        raise ValueError("Unsupported correlation method")

    return df[column1].corr(df[column2], method=method)


# Usage
df = pd.read_csv("airlines.csv")
correlation = get_correlation(df, "AirportFrom", "AirportTo")

print(f"Correlation: {correlation:.4f}")

Correlation: 0.0151


# Fitness of Normal Distribution
TODO: Maybe add different distributions

In [21]:
from scipy import stats
import pandas as pd
import numpy as np

def test_normality(df, column, alpha=0.05):
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found")

    if not pd.api.types.is_numeric_dtype(df[column].dtype):
        raise TypeError(f"Column '{column}' must be numeric")

    values = df[column].dropna().to_numpy()

    if len(values) < 8:
        raise ValueError("At least 8 observations are required")

    if not np.isfinite(values).all():
        raise ValueError("Column contains infinite values")

    if np.ptp(values) == 0:
        raise ValueError("Cannot test a constant column")

    result = stats.normaltest(values)

    return {
        "statistic": result.statistic,
        "p_value": result.pvalue,
        "is_normal": result.pvalue >= alpha
    }


# Usage
df = pd.read_csv("airlines.csv")
result = test_normality(df, "AirportFrom")

print(result)

{'statistic': np.float64(388410.01508424437), 'p_value': np.float64(0.0), 'is_normal': np.False_}


# General algorithm
The idea for data generation is to retrieve the metrics of the data first and to provide margins on how far we can deviate from the original distribution during trace-based data generation. Our goal is to generate the data beforehand, so we can test data before a benchmark run, meaning we can create a validation loop where we can prioritize different metrics during each generation step and build progresssively a new data set.
For instance: We might want to increase null ratios to challenge join estimations. Before that, we would maintain the original average value of a column and its variance. ~~We can normalize the whole data set after introdcuing the null values randomly by adding or substracting the difference in estimated value~~. We would need a more sophisticated algortihm to account for updating variance and covariance. There are ways to calulate the update variance and covariance (Welford algorithm?), which we could maybe use to create pairs of data that maintain the generated data during generation. In another step, we can then remove data that have the least amount of impact on these properties to then restore the amount of entries and Distinct Values that were originally present, while respecting the data range
We might have to adjust the exact validation loop dependent on the type of transformation we want to apply

<b>TODO:</b> What about Fitness of distribution? Are there less invasie methods to retain data? We also still miss a reliable way to measure cross-column relations properly -> Postgresql does concern itself with those measurements, might be a good start to add new parameters